# Unraveling Network Signatures of Viral Oncogenicity

This notebook implements the analytical pipeline described in the paper: **"Unraveling the Network Signatures of Oncogenicity in Virus–Human Protein–Protein Interactions"** (Zambelli et al., *Entropy* 2025).

## 1. Project Overview
The core hypothesis of this research is that oncogenic viruses—which account for ~15% of global cancer cases—perturb the human interactome in a structurally distinct way compared to non-oncogenic viruses. By representing these interactions as **multilayer networks**, we can identify shared topological signatures that define the oncogenic potential of a virus.

### Why Multilayer Networks?
Instead of looking at each virus in isolation, we treat each virus-host interaction network as a layer in a multiplex structure. This allows us to observe **collective properties** that emerge only when multiple viral perturbations are analyzed together, effectively uncovering common strategies used by oncogenic viruses to hijack human cellular machinery.

In [ ]:
import sys
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Suppress library noise (e.g., graph-tool warnings)
warnings.filterwarnings("ignore", category=UserWarning)

# Add src directory to path
sys.path.append(os.path.abspath("src"))

print("Environment initialized and modules ready.")

## 2. Step 1: Reconstructing Viral Impact Zones

For each of the 80 viruses in our dataset (8 oncogenic, 72 non-oncogenic), we reconstruct a subnetwork from the **human interactome (BioGRID/STRING)**.

### Method:
1. **Primary Targets**: Identify human proteins directly targeted by viral proteins.
2. **Functional Neighborhood**: Include the first neighbors (direct interactors) of these targets.
3. **Subgraph Extraction**: Subset the human interactome to only these nodes, creating a "proxy" for the region of the cell most influenced by the infection.

**Biological Interpretation**: This process identifies the "battlefield" where viral proteins interact with host machinery, focusing on the most immediate functional perturbations.

In [ ]:
from network_factory import generate_all_viral_networks

# Reconstruct networks for all 80 viral species
generate_all_viral_networks()

## 3. Step 2: Multilayer Network Composition

We group the 80 viruses into 4-layer multiplex networks. This size (4 layers) provides a balance between computational feasibility and the ability to detect statistically significant patterns.

### Combination Sets:
To compare oncogenic vs. non-oncogenic signatures, we generate random combinations across five sets:
- **N**: 4 Non-oncogenic layers (Control).
- **N1O**: 3 Non-oncogenic + 1 Oncogenic.
- **N2O**: 2 Non-oncogenic + 2 Oncogenic.
- **N3O**: 1 Non-oncogenic + 3 Oncogenic.
- **O**: 4 Oncogenic layers (Target).

We use **Multilayer PageRank Versatility** to rank nodes across these layers, identifying proteins that attract the most information flow across the entire multiplex system.

In [ ]:
from analyzer import generate_combination_indexes

# Generate random combination indexes (default 256 iterations per set)
generate_combination_indexes(n_iters=5) # Reduced for demo

## 4. Step 3: Extracting Topological Signatures

This section implements the core topological analysis. We extract four key features from each multilayer network, each carrying a specific biological meaning.

### A. Largest Viable Component (LVC)
- **Description**: The maximum subset of nodes connected by the same path in all layers independently.
- **Biological Interpretation**: The LVC represents the **functional core** shared by the viruses. Oncogenic viruses exhibit larger LVCs, indicating they target overlapping, critical regions of the interactome. These cores are highly enriched for **TP53, MDM2, PARP1**, and chromatin remodeling pathways.

### B. Targeted Percolation (Critical Point)
- **Description**: Simulates a "targeted attack" on the network based on PageRank versatility. The critical point is the fraction of nodes that must be removed to collapse the largest connected component.
- **Biological Interpretation**: Oncogenic networks have higher critical points, meaning they are **more robust**. This suggests that oncogenic viruses target central, evolutionary ancient processes (e.g., cell growth, transcription) that are more densely connected and essential to life.

### C. Modular Structure (SBM)
- **Description**: We use the **Degree-Corrected Stochastic Block Model (DCSBM)** to infer communities (modules) at the mesoscale.
- **Biological Interpretation**: Oncogenic networks tend to have **more modules** but **lower modularity**. This signals a **structural disruption**—oncogenic viruses fragment the functional organization of the cell, leading to a more homogeneous (less specialized) distribution of links, which is a hallmark of systemic cellular dysfunction in cancer.

In [ ]:
from analyzer import run_topological_analysis

# Execute LVC calculation, Percolation analysis, and SBM inference
run_topological_analysis()

## 5. Step 4: Machine Learning Classification

We combine the topological features into a 4D vector to classify the multilayer networks.

### Dimensionality Reduction (UMAP)
We use **UMAP** to project the high-dimensional topological space into 2D. UMAP is chosen for its ability to preserve both local and global structure, capturing non-linear relationships between the features.

### Classification (SVM)
A **Support Vector Machine (SVM)** with a Gaussian kernel is used to define the boundary between oncogenic and non-oncogenic regions. 

**Key Finding**: As the proportion of oncogenic layers in a multiplex network increases, the samples clearly shift from the "non-oncogenic" to the "oncogenic" cluster in the UMAP space, confirming that topological features alone are sufficient to recognize oncogenic signatures.

In [ ]:
from statistics import load_results, plot_distributions, run_ml_classification

# Load data from the 'results' directory
lvcs, crit_points, mods, mody = load_results()

if lvcs:
    # Generate statistical boxplots for LVC, Critical Point, and Modularity
    plot_distributions(lvcs, crit_points, mods, mody)
    
    # Run the UMAP projection and SVM classification
    run_ml_classification(lvcs, crit_points, mods, mody)
else:
    print("Waiting for Step 3 to produce result files.")

## 6. Conclusions
The pipeline demonstrates that oncogenicity is not just a molecular property of single viral proteins, but a **topological signature** that can be identified at the system level. The shift toward higher robustness and lower modularity in oncogenic networks highlights a systematic strategy of attacking the core, evolutionary-conserved functions of the human host cell.